# 04 Customer Segmentation Analysis

## Objective

This notebook analyzes the Gold layer created for the Olist Customer Analytics project.

The goal is to transform customer-level features and RFM segments into business insights about customer value, retention, delivery experience, and satisfaction.

Main analytical themes:

- Customer base overview.
- One-time versus repeat customers.
- Customer segment distribution.
- Revenue and value by segment.
- Delivery experience by segment.
- Satisfaction and review score analysis.
- Business insights and recommendations.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

## 2. Define paths

This notebook assumes it is located inside:

`01_customer_analytics/notebooks/`

Therefore, `..` points to the `01_customer_analytics/` project folder.

In [ ]:
PROJECT_PATH = Path("..")
GOLD_PATH = PROJECT_PATH / "data" / "gold"

GOLD_PATH

## 3. Load Gold datasets

In [ ]:
fact_orders = pd.read_parquet(GOLD_PATH / "fact_orders.parquet")
fact_order_items = pd.read_parquet(GOLD_PATH / "fact_order_items.parquet")
customer_features = pd.read_parquet(GOLD_PATH / "customer_features.parquet")
customer_rfm = pd.read_parquet(GOLD_PATH / "customer_rfm.parquet")
customer_segments = pd.read_parquet(GOLD_PATH / "customer_segments.parquet")

fact_orders.shape, customer_features.shape, customer_segments.shape

## 4. Create analytical base

In [ ]:
customer_analysis = (
    customer_features
    .merge(
        customer_segments[[
            "customer_unique_id",
            "recency_days",
            "frequency",
            "monetary_value",
            "r_score",
            "f_score",
            "m_score",
            "rfm_score",
            "customer_segment"
        ]],
        on="customer_unique_id",
        how="left"
    )
)

customer_analysis.head()

## 5. Customer base overview

In [ ]:
total_customers = customer_analysis["customer_unique_id"].nunique()
total_orders = fact_orders["order_id"].nunique()
total_revenue = fact_orders["total_order_value"].sum()
avg_order_value = fact_orders["total_order_value"].mean()
avg_orders_per_customer = total_orders / total_customers

overview = pd.DataFrame({
    "metric": [
        "Total customers",
        "Total orders",
        "Total revenue",
        "Average order value",
        "Average orders per customer"
    ],
    "value": [
        total_customers,
        total_orders,
        total_revenue,
        avg_order_value,
        avg_orders_per_customer
    ]
})

overview

## 6. One-time versus repeat customers

In [ ]:
customer_analysis["customer_type"] = np.where(
    customer_analysis["total_orders"] == 1,
    "One-time buyer",
    "Repeat buyer"
)

customer_type_summary = (
    customer_analysis
    .groupby("customer_type", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        total_revenue=("total_spent", "sum"),
        avg_customer_value=("total_spent", "mean"),
        avg_orders=("total_orders", "mean"),
        avg_review_score=("avg_review_score", "mean"),
        delay_rate=("delay_rate", "mean")
    )
)

customer_type_summary["customer_share"] = (
    customer_type_summary["customers"] / customer_type_summary["customers"].sum()
)

customer_type_summary["revenue_share"] = (
    customer_type_summary["total_revenue"] / customer_type_summary["total_revenue"].sum()
)

customer_type_summary.sort_values("customers", ascending=False)

### Insight placeholder

Use this space to describe what the one-time versus repeat customer split means for the business.

## 7. Customer segment distribution

In [ ]:
segment_distribution = (
    customer_analysis
    .groupby("customer_segment", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        total_revenue=("total_spent", "sum"),
        avg_customer_value=("total_spent", "mean"),
        avg_order_value=("avg_order_value", "mean"),
        avg_orders=("total_orders", "mean"),
        avg_recency_days=("recency_days", "mean"),
        avg_review_score=("avg_review_score", "mean"),
        delay_rate=("delay_rate", "mean")
    )
)

segment_distribution["customer_share"] = (
    segment_distribution["customers"] / segment_distribution["customers"].sum()
)

segment_distribution["revenue_share"] = (
    segment_distribution["total_revenue"] / segment_distribution["total_revenue"].sum()
)

segment_distribution.sort_values("customers", ascending=False)

## 8. Visualize segment distribution

In [ ]:
plot_df = segment_distribution.sort_values("customers", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["customer_segment"], plot_df["customers"])
plt.title("Customers by Segment")
plt.xlabel("Customers")
plt.ylabel("Customer Segment")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = segment_distribution.sort_values("total_revenue", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["customer_segment"], plot_df["total_revenue"])
plt.title("Revenue by Segment")
plt.xlabel("Total Revenue")
plt.ylabel("Customer Segment")
plt.tight_layout()
plt.show()

## 9. Segment value comparison

In [ ]:
segment_value_summary = (
    segment_distribution
    .sort_values("avg_customer_value", ascending=False)
    [[
        "customer_segment",
        "customers",
        "avg_customer_value",
        "avg_order_value",
        "avg_orders",
        "revenue_share"
    ]]
)

segment_value_summary

## 10. Delivery experience analysis

In [ ]:
delivery_summary = pd.DataFrame({
    "metric": [
        "Average delivery days",
        "Average estimated delivery days",
        "Average delivery delay days",
        "Delayed order rate"
    ],
    "value": [
        fact_orders["delivery_days"].mean(),
        fact_orders["estimated_delivery_days"].mean(),
        fact_orders["delivery_delay_days"].mean(),
        fact_orders["is_delayed"].mean()
    ]
})

delivery_summary

In [ ]:
delivery_by_segment = (
    customer_analysis
    .groupby("customer_segment", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        avg_delivery_days=("avg_delivery_days", "mean"),
        delay_rate=("delay_rate", "mean"),
        avg_review_score=("avg_review_score", "mean")
    )
    .sort_values("delay_rate", ascending=False)
)

delivery_by_segment

## 11. Satisfaction analysis

In [ ]:
review_summary = pd.DataFrame({
    "metric": [
        "Average review score",
        "Median review score",
        "Share of reviews <= 2",
        "Share of reviews = 5"
    ],
    "value": [
        fact_orders["review_score"].mean(),
        fact_orders["review_score"].median(),
        (fact_orders["review_score"] <= 2).mean(),
        (fact_orders["review_score"] == 5).mean()
    ]
})

review_summary

In [ ]:
review_by_delay = (
    fact_orders
    .assign(delivery_status=np.where(fact_orders["is_delayed"] == 1, "Delayed", "On time"))
    .groupby("delivery_status", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        low_review_share=("review_score", lambda x: (x <= 2).mean()),
        five_star_share=("review_score", lambda x: (x == 5).mean())
    )
)

review_by_delay

In [ ]:
plot_df = review_by_delay.sort_values("avg_review_score", ascending=True)

plt.figure(figsize=(8, 5))
plt.bar(plot_df["delivery_status"], plot_df["avg_review_score"])
plt.title("Average Review Score by Delivery Status")
plt.xlabel("Delivery Status")
plt.ylabel("Average Review Score")
plt.ylim(0, 5)
plt.tight_layout()
plt.show()

## 12. Geographic analysis

In [ ]:
state_summary = (
    customer_analysis
    .groupby("customer_state", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        total_revenue=("total_spent", "sum"),
        avg_customer_value=("total_spent", "mean"),
        avg_review_score=("avg_review_score", "mean"),
        delay_rate=("delay_rate", "mean")
    )
    .sort_values("customers", ascending=False)
)

state_summary.head(10)

## 13. Category analysis

In [ ]:
category_summary = (
    fact_order_items
    .merge(
        fact_orders[["order_id", "review_score", "is_delayed", "customer_unique_id"]],
        on="order_id",
        how="left"
    )
    .groupby("product_category_name_english", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique"),
        revenue=("price", "sum"),
        avg_item_price=("price", "mean"),
        avg_review_score=("review_score", "mean"),
        delay_rate=("is_delayed", "mean")
    )
    .sort_values("revenue", ascending=False)
)

category_summary.head(10)

## 14. Business insights draft

Use the outputs above to write final insights. Suggested structure:

1. **Customer base is highly transactional.**  
   A large share of customers made only one purchase, which makes retention and reactivation critical topics.

2. **High-value one-time buyers deserve dedicated attention.**  
   Some one-time buyers generate meaningful revenue despite not repeating purchases.

3. **Customer segments differ in value and experience.**  
   Segment-level revenue, review score, and delay rate can support different CRM actions.

4. **Delivery experience is a relevant satisfaction lever.**  
   Comparing review scores between delayed and on-time orders helps quantify the impact of operations on customer experience.

5. **Geography and category performance can guide operational priorities.**  
   States and product categories with higher delay rates or lower review scores may require specific interventions.


## 15. Recommended next steps

Recommended next steps for the project:

- Refine the RFM segmentation rules if needed.
- Export segment-level summary tables for Power BI.
- Build the Power BI dashboard.
- Update the README with methodology, data model, insights, and business recommendations.
